In [ ]:
# rag_system.py
"""
LangChain v0.3 RAG System (class-oriented) using ChromaDB.

Features:
 - Loads hierarchical JSON files under json_input_root (expects structure: json_data/dbe_<TICKER>/*.json).
 - Ingests into parent documents + child chunks (paragraphs, tables).
 - Multi-representation indexing using MultiVectorRetriever (child chunk vectors -> parent doc store).
 - Additional retrievers: dense (chroma), MultiQueryRetriever, MergerRetriever, HyDE, Decomposition, Step-Back.
 - Prompt versioning (PromptManager) and experiment logging (ExperimentLogger).
 - Evaluation helpers (EM / F1).
"""

import os
import json
import uuid
import time
import logging
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from datetime import datetime

# LangChain v0.3 imports (see docs)
from langchain.chat_models import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.schema import Document as LCDocument
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers.merger_retriever import MergerRetriever
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import InMemoryByteStore

# small local sparse retriever for complementing dense retrieval
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# -------------------------
# Logging & Configuration
# -------------------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")


class RAGConfig:
    """Edit these defaults to suit your environment."""
    JSON_INPUT_ROOT: Path = Path("./json_data")          # expects json_data/dbe_<TICKER>/*.json
    CHROMA_PERSIST_DIR: Path = Path("./chroma_store")   # directory for Chroma persistence
    CHROMA_COLLECTION_NAME: str = "ragg_child_chunks"
    SUMMARIES_COLLECTION: str = "ragg_summaries"        # used for MultiVectorRetriever summary vectors
    PROMPTS_DIR: Path = Path("./prompts")
    EXPERIMENTS_DIR: Path = Path("./experiments")
    MODEL_NAME: str = "gpt-4o-mini"                      # adjust if you use another OpenAI model
    EMBEDDING_MODEL: str = "text-embedding-3-small"
    TEMPERATURE: float = 0.0
    K: int = 6                                           # default retrieve size
    CHUNK_SIZE: int = 1000                               # characters for splitting paragraphs (tune)
    CHUNK_OVERLAP: int = 200

    def __post_init__(self):
        self.PROMPTS_DIR.mkdir(parents=True, exist_ok=True)
        self.EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)
        self.CHROMA_PERSIST_DIR.mkdir(parents=True, exist_ok=True)


# -------------------------
# Prompt Manager & Experiment Logger
# -------------------------
class PromptManager:
    """Register and load prompt versions locally (json files)."""
    def __init__(self, prompts_dir: Path):
        self.prompts_dir = prompts_dir
        self.prompts_dir.mkdir(parents=True, exist_ok=True)

    def register_prompt(self, name: str, template: str, description: str = "") -> str:
        pid = f"{name.replace(' ', '_')}_{int(time.time())}"
        payload = {
            "id": pid,
            "name": name,
            "template": template,
            "description": description,
            "created_at": datetime.utcnow().isoformat()
        }
        path = self.prompts_dir / f"{pid}.json"
        path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
        logging.info(f"Prompt registered: {pid}")
        return pid

    def load_prompt(self, prompt_id: str) -> Dict[str, Any]:
        p = self.prompts_dir / f"{prompt_id}.json"
        if not p.exists():
            raise FileNotFoundError(f"Prompt id {prompt_id} not found at {p}")
        return json.loads(p.read_text(encoding="utf-8"))

    def list_prompts(self) -> List[str]:
        return [p.stem for p in sorted(self.prompts_dir.glob("*.json"))]


class ExperimentLogger:
    """Logs experiments to individual JSON files + index.jsonl"""
    def __init__(self, experiments_dir: Path):
        self.dir = experiments_dir
        self.dir.mkdir(parents=True, exist_ok=True)
        self.index = self.dir / "index.jsonl"

    def log(self, entry: Dict[str, Any]) -> str:
        run_id = f"run_{int(time.time()*1000)}_{uuid.uuid4().hex[:6]}"
        payload = {"run_id": run_id, "timestamp": datetime.utcnow().isoformat(), **entry}
        run_path = self.dir / f"{run_id}.json"
        run_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
        with self.index.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(payload, ensure_ascii=False) + "\n")
        logging.info(f"Experiment logged: {run_id}")
        return run_id


# -------------------------
# Simple TF-IDF Retriever (sparse complement)
# -------------------------
class TfidfRetriever:
    """Small TF-IDF retriever returning LangChain Documents (keeps metadata)."""
    def __init__(self, docs: List[LCDocument], k: int = 5):
        self.docs = docs
        self.k = k
        self.texts = [d.page_content for d in docs]
        self.vectorizer = TfidfVectorizer(stop_words="english", max_features=20000)
        if len(self.texts) == 0:
            self.mat = None
        else:
            self.mat = self.vectorizer.fit_transform(self.texts)

    def get_relevant_documents(self, query: str) -> List[LCDocument]:
        if self.mat is None:
            return []
        v = self.vectorizer.transform([query])
        scores = (self.mat @ v.T).toarray().ravel()
        idx = np.argsort(scores)[::-1][: self.k]
        # return top-k (even if zero scores) but preserve docs length
        return [self.docs[i] for i in idx]


# -------------------------
# RAG System Class
# -------------------------
class RAGSystem:
    def __init__(self, cfg: RAGConfig):
        self.cfg = cfg
        # LLM + embeddings
        self.llm = ChatOpenAI(model=self.cfg.MODEL_NAME, temperature=self.cfg.TEMPERATURE)
        self.embeddings = OpenAIEmbeddings(model=self.cfg.EMBEDDING_MODEL)

        # managers
        self.prompt_manager = PromptManager(self.cfg.PROMPTS_DIR)
        self.experiment_logger = ExperimentLogger(self.cfg.EXPERIMENTS_DIR)

        # internal state holders
        self.parent_docs: List[LCDocument] = []   # original documents (parent)
        self.child_docs: List[LCDocument] = []    # splitted/child chunks (paragraphs, flattened tables)
        self.doc_id_map: Dict[str, LCDocument] = {}  # doc_id -> parent LCDocument

        # vectorstores and retrievers (built in build_indexes)
        self.child_vectorstore: Optional[Chroma] = None
        self.summaries_vectorstore: Optional[Chroma] = None
        self.multi_vector_retriever: Optional[MultiVectorRetriever] = None
        self.dense_retriever = None
        self.multiquery_retriever = None
        self.merger_retriever = None
        self.tfidf_retriever = None

        # text splitters
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.cfg.CHUNK_SIZE,
            chunk_overlap=self.cfg.CHUNK_OVERLAP
        )

    # ---------- Ingest hierarchical JSON -> parent + child docs ----------
    def ingest_documents(self, json_root: Path):
        """
        Walks json_root/dbe_<TICKER>/*.json and builds:
         - parent_docs: each json file represented as a parent document (full_text).
         - child_docs: smaller chunks with metadata and doc_id linking to parent.
        """
        json_root = Path(json_root)
        if not json_root.exists():
            raise FileNotFoundError(f"{json_root} does not exist")

        self.parent_docs = []
        self.child_docs = []
        id_key = "doc_id"

        for dbe_dir in sorted(json_root.glob("dbe_*")):
            if not dbe_dir.is_dir():
                continue
            ticker = dbe_dir.name.replace("dbe_", "", 1)
            for jfile in sorted(dbe_dir.glob("*.json")):
                data = json.loads(jfile.read_text(encoding="utf-8"))
                file_label = data.get("file", jfile.name)
                # create consolidated parent content (concatenate sections, subsections)
                parts = []
                sections = data.get("sections") or data.get("hierarchy") or []
                for sec in sections:
                    sec_title = sec.get("title") or sec.get("section") or ""
                    parts.append(f"SECTION: {sec_title}")
                    # any paragraphs at section level
                    for p in sec.get("paragraphs", []):
                        parts.append(p)
                    # subsections
                    for sub in sec.get("subsections", []):
                        sub_title = sub.get("title") or sub.get("subsection") or ""
                        parts.append(f"SUBSECTION: {sub_title}")
                        for p in sub.get("paragraphs", []):
                            parts.append(p)
                        # tables: represent as readable flattened text now (keeps rows/cols)
                        for tbl in sub.get("tables", []):
                            parts.append(self._flatten_table_text(tbl))
                    # tables attached to section
                    for tbl in sec.get("tables", []):
                        parts.append(self._flatten_table_text(tbl))

                full_text = "\n\n".join(parts).strip() or ""
                # generate doc_id for parent doc
                doc_id = str(uuid.uuid4())
                parent_md = {
                    "ticker": ticker,
                    "file": file_label,
                    "source_path": str(jfile),
                    "doc_id": doc_id
                }
                parent_doc = LCDocument(page_content=full_text, metadata=parent_md)
                self.parent_docs.append(parent_doc)
                self.doc_id_map[doc_id] = parent_doc

                # create child docs from the structure (prefer preserving natural paragraphs and tables)
                # --- paragraphs and tables at different granularities are added as child docs with metadata referencing doc_id
                idx_counter = 0
                for sec in sections:
                    sec_title = sec.get("title") or sec.get("section") or ""
                    # paragraphs at section
                    for p in sec.get("paragraphs", []):
                        child_md = dict(parent_md)
                        child_md.update({
                            id_key: doc_id,
                            "section_title": sec_title,
                            "subsection_title": None,
                            "is_table": False,
                            "child_index": idx_counter
                        })
                        idx_counter += 1
                        self.child_docs.append(LCDocument(page_content=p, metadata=child_md))
                    # subsections
                    for sub in sec.get("subsections", []):
                        sub_title = sub.get("title") or sub.get("subsection") or ""
                        for p in sub.get("paragraphs", []):
                            child_md = dict(parent_md)
                            child_md.update({
                                id_key: doc_id,
                                "section_title": sec_title,
                                "subsection_title": sub_title,
                                "is_table": False,
                                "child_index": idx_counter
                            })
                            idx_counter += 1
                            self.child_docs.append(LCDocument(page_content=p, metadata=child_md))
                        for t in sub.get("tables", []):
                            tbl_text = self._flatten_table_text(t)
                            child_md = dict(parent_md)
                            child_md.update({
                                id_key: doc_id,
                                "section_title": sec_title,
                                "subsection_title": sub_title,
                                "is_table": True,
                                "table_title": t.get("title"),
                                "child_index": idx_counter
                            })
                            idx_counter += 1
                            self.child_docs.append(LCDocument(page_content=tbl_text, metadata=child_md))
                    # tables at section level
                    for t in sec.get("tables", []):
                        tbl_text = self._flatten_table_text(t)
                        child_md = dict(parent_md)
                        child_md.update({
                            id_key: doc_id,
                            "section_title": sec_title,
                            "subsection_title": None,
                            "is_table": True,
                            "table_title": t.get("title"),
                            "child_index": idx_counter
                        })
                        idx_counter += 1
                        self.child_docs.append(LCDocument(page_content=tbl_text, metadata=child_md))

        logging.info(f"Ingested parent_docs={len(self.parent_docs)}, child_docs={len(self.child_docs)}")

    def _flatten_table_text(self, table: Dict[str, Any]) -> str:
        """Create a readable textual/table representation preserving header->values.
           This representation is meant for embedding and downstream LLM consumption.
        """
        lines = []
        title = table.get("title") or ""
        if title:
            lines.append(f"TABLE: {title}")

        # try several possible table shapes used by your earlier parser
        # 1) 'columns' + 'rows' where rows are dicts with 'cells' or list
        columns = table.get("columns") or table.get("headers") or []
        if columns:
            # column labels
            col_labels = [c.get("label") if isinstance(c, dict) else str(c) for c in columns]
            lines.append("COLUMNS: " + " | ".join(col_labels))

        rows = table.get("rows") or table.get("data") or []
        for r in rows:
            if isinstance(r, dict):
                label = r.get("label") or r.get("row_header") or ""
                # if row has 'cells' dict
                cells = r.get("cells") or r.get("values") or {}
                if isinstance(cells, dict):
                    kvs = [f"{k}: {v}" for k, v in cells.items()]
                    lines.append(f"{label} -> " + "; ".join(kvs))
                elif isinstance(cells, list):
                    lines.append(f"{label} -> " + " | ".join([str(x) for x in cells]))
                else:
                    # fallback
                    lines.append(json.dumps(r, ensure_ascii=False))
            elif isinstance(r, list):
                # row list (aligned with columns)
                lines.append(" | ".join([str(x) for x in r]))
            else:
                lines.append(str(r))
        return "\n".join(lines)

    # ---------- Build/chroma indexes + retrievers ----------
    def build_indexes(self, persist: bool = True):
        """Builds the Chroma vectorstore for child docs and a MultiVectorRetriever mapping child vectors -> parent docs."""
        # sanity
        if not self.child_docs or not self.parent_docs:
            raise RuntimeError("No documents ingested. Run ingest_documents() first.")

        # 1) Create child vectorstore (chunks)
        # NOTE: Chroma initialization depends on langchain_chroma integration installed
        child_vs = Chroma(
            collection_name=self.cfg.CHROMA_COLLECTION_NAME,
            embedding_function=self.embeddings,
            persist_directory=str(self.cfg.CHROMA_PERSIST_DIR)
        )
        # add child docs to vectorstore (this will compute embeddings)
        child_vs.add_documents(self.child_docs)
        if persist:
            child_vs.persist()
        self.child_vectorstore = child_vs
        logging.info("Child vectorstore built and persisted.")

        # 2) Dense retriever (simple as_retriever)
        self.dense_retriever = child_vs.as_retriever(search_kwargs={"k": self.cfg.K})

        # 3) TF-IDF retriever (sparse)
        self.tfidf_retriever = TfidfRetriever(self.child_docs, k=self.cfg.K)

        # 4) MultiQuery retriever (uses LLM to expand queries)
        try:
            self.multiquery_retriever = MultiQueryRetriever.from_llm(
                retriever=child_vs.as_retriever(search_kwargs={"k": self.cfg.K}),
                llm=self.llm
            )
        except Exception:
            # fallback: use dense retriever
            self.multiquery_retriever = self.dense_retriever

        # 5) Merger retriever (merges dense + sparse)
        self.merger_retriever = MergerRetriever(retrievers=[self.dense_retriever, self.tfidf_retriever])

        # 6) MultiVectorRetriever (RAPTOR-style) -> use child vectorstore as vector DB for summary vectors
        # We'll create a second Chroma collection to store summaries / hypothetical questions mapped to parent docs.
        summaries_vs = Chroma(
            collection_name=self.cfg.SUMMARIES_COLLECTION,
            embedding_function=self.embeddings,
            persist_directory=str(self.cfg.CHROMA_PERSIST_DIR)  # same DB dir, different collection name
        )
        # Build simple summaries (one per parent_doc) by summarizing the parent content using the LLM (batched)
        summaries = []
        doc_ids = []
        for parent in self.parent_docs:
            doc_id = parent.metadata.get("doc_id")
            doc_ids.append(doc_id)
            # small summarization prompt (we do one LLM call per parent; you can batch/parallelize)
            summarizer_template = (
                "Summarize the following filing in 2-3 short sentences focusing on key financial tables and amounts:\n\n"
                "{doc}"
            )
            prompt = ChatPromptTemplate.from_template(summarizer_template)
            chain = prompt | self.llm
            try:
                summary = chain.invoke({"doc": parent.page_content})
            except Exception:
                # fallback to truncated text
                summary = parent.page_content[:1000]
            # ensure string
            summary_text = str(summary).strip()
            summaries.append(LCDocument(page_content=summary_text, metadata={"doc_id": doc_id}))
        # add summaries into summaries vectorstore
        summaries_vs.add_documents(summaries)
        if persist:
            summaries_vs.persist()
        self.summaries_vectorstore = summaries_vs

        # build an in-memory parent docstore and populate it (required by MultiVectorRetriever)
        byte_store = InMemoryByteStore()
        # docstore expects (id, Document) pairs
        parent_pairs = [(p.metadata.get("doc_id"), p) for p in self.parent_docs]
        byte_store.mset(parent_pairs)

        # create MultiVectorRetriever that maps child/summaries vectors to parent docs
        mv_retriever = MultiVectorRetriever(
            vectorstore=summaries_vs,
            byte_store=byte_store,
            id_key="doc_id",
            search_kwargs={"k": self.cfg.K}
        )
        # note: we could add other vectors (hypothetical questions) to the same summaries_vs collection if desired
        self.multi_vector_retriever = mv_retriever

        logging.info("Indexes and retrievers built: dense, multi-query, merger, multi-vector, tfidf (sparse).")

    # ---------- Retriever variants ----------
    def _hyde_retrieve(self, question: str, k: Optional[int] = None) -> List[LCDocument]:
        """HyDE: generate a hypothetical answer paragraph, embed it, then similarity search on child vectorstore."""
        k = k or self.cfg.K
        hyde_prompt = ChatPromptTemplate.from_template(
            "Write a short factual paragraph that directly answers the question (no sources). Question: {question}"
        )
        chain = hyde_prompt | self.llm
        hyde_text = chain.invoke({"question": question})
        hyde_text = str(hyde_text)
        # embed query and similarity search by vector
        vec = self.embeddings.embed_query(hyde_text)
        docs = self.child_vectorstore.similarity_search_by_vector(vec, k=k)
        return docs

    def _decomposition_retrieve(self, question: str, k_each: int = 4) -> List[LCDocument]:
        """Ask the LLM to decompose the question into sub-questions, retrieve for each, union results."""
        subq_prompt = ChatPromptTemplate.from_template(
            "Decompose the question into 3-5 focused sub-questions (one per line):\n\nQuestion: {question}"
        )
        chain = subq_prompt | self.llm
        subq_text = chain.invoke({"question": question})
        subq_text = str(subq_text)
        subqs = [s.strip("-. \t") for s in subq_text.splitlines() if s.strip()]
        all_docs = []
        seen = set()
        for sq in subqs:
            docs = self.dense_retriever.get_relevant_documents(sq) if hasattr(self.dense_retriever, "get_relevant_documents") else self.dense_retriever(sq)
            for d in docs:
                key = (d.metadata.get("source_path"), d.page_content[:200])
                if key not in seen:
                    all_docs.append(d)
                    seen.add(key)
        return all_docs

    # ---------- Answer synthesis ----------
    def _synthesize_answer(self, docs: List[LCDocument], question: str, prompt_template: Optional[str] = None) -> Tuple[str, List[Dict[str, Any]]]:
        """Produce final answer with LLM and return (answer_text, provenance)."""
        # default prompt
        if prompt_template is None:
            prompt_template = (
                "You are an expert financial assistant. Use the context below to answer the question concisely and factually.\n\n"
                "Context:\n{context}\n\nQuestion: {question}\n\nAnswer (short, cite sources inline as [Source X]):"
            )
        # build context string with provenance headers up to top-K
        context_lines = []
        for i, d in enumerate(docs[: self.cfg.K ]):
            md = d.metadata or {}
            src = md.get("source_path") or md.get("file") or md.get("ticker") or "unknown"
            header = f"[Source {i+1}] {src} (ticker={md.get('ticker')}, section={md.get('section_title')}, subsection={md.get('subsection_title')})"
            context_lines.append(header + "\n" + d.page_content)
        context_block = "\n\n---\n\n".join(context_lines) or "No context found."

        prompt = ChatPromptTemplate.from_template(prompt_template)
        chain = prompt | self.llm
        try:
            out = chain.invoke({"context": context_block, "question": question})
            answer = str(out).strip()
        except Exception as ex:
            # fallback: make a simple call
            answer = f"(LLM error): {ex}"

        provenance = [{"source": (d.metadata.get("source_path") or d.metadata.get("file")), "metadata": d.metadata} for d in docs]
        return answer, provenance

    # ---------- Routing & Orchestration ----------
    def route_strategy(self, question: str, options: List[str]) -> str:
        """Ask LLM to pick a strategy name from options (deterministic intent, temp=0)."""
        prompt = (
            "You are a routing assistant. Choose exactly one best retrieval strategy (return only the exact name):\n\n"
            f"Options: {', '.join(options)}\n\nQuestion: {question}\n\nReturn one option name exactly."
        )
        chain = ChatPromptTemplate.from_template("{q}") | self.llm
        choice = chain.invoke({"q": prompt})
        choice = str(choice).strip()
        # validate
        if choice not in options:
            # attempt matching substring
            for o in options:
                if o.lower() in choice.lower():
                    return o
            return options[0]
        return choice

    def run_query(self, question: str, strategy: Optional[str] = None, prompt_id: Optional[str] = None, extra: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        """
        High-level entry to run a single question.
        strategy options: ['naive','multi_query','fusion','hyde','decompose','step_back','raptor','multi_vector','auto']
        If prompt_id is provided, the prompt template from PromptManager is used for synthesis.
        """
        extra = extra or {}
        strategies = ['naive', 'multi_query', 'fusion', 'hyde', 'decompose', 'step_back', 'raptor', 'multi_vector']
        if strategy is None or strategy == "auto":
            chosen = self.route_strategy(question, strategies)
            logging.info(f"Router chose strategy: {chosen}")
            strategy = chosen

        prompt_template = None
        prompt_version = "default"
        if prompt_id:
            info = self.prompt_manager.load_prompt(prompt_id)
            prompt_template = info.get("template")
            prompt_version = prompt_id

        # retrieval stage
        docs = []
        if strategy == "naive":
            docs = self.dense_retriever.get_relevant_documents(question)
        elif strategy == "multi_query":
            docs = self.multiquery_retriever.get_relevant_documents(question)
        elif strategy == "fusion":
            # MergerRetriever: merges dense + tfidf
            docs = self.merger_retriever.get_relevant_documents(question)
        elif strategy == "hyde":
            docs = self._hyde_retrieve(question)
        elif strategy == "decompose":
            docs = self._decomposition_retrieve(question)
        elif strategy == "step_back":
            # initial answer with naive, then follow-ups
            initial_docs = self.dense_retriever.get_relevant_documents(question)
            answer0, _ = self._synthesize_answer(initial_docs, question, prompt_template)
            followup_prompt = ChatPromptTemplate.from_template(
                "You produced the following answer:\n\n{ans}\n\nGenerate up to 3 specific follow-up queries that would help obtain missing evidence (one per line)."
            )
            chain = followup_prompt | self.llm
            followup_text = chain.invoke({"ans": answer0})
            followups = [l.strip("-. \t") for l in str(followup_text).splitlines() if l.strip()]
            extra_docs = []
            for fq in followups:
                extra_docs.extend(self.dense_retriever.get_relevant_documents(fq))
            # union & dedupe top
            seen = set(); merged=[]
            for d in (initial_docs + extra_docs):
                key = (d.metadata.get("source_path"), d.page_content[:200])
                if key not in seen:
                    merged.append(d); seen.add(key)
            docs = merged
        elif strategy == "raptor":
            # RAPTOR-style hierarchical retrieval:
            # 1) retrieve parent docs via MultiVectorRetriever (summaries/hypo)
            # 2) for top N parent docs, fetch child chunks and run dense retrieval restricted to those doc_ids
            top_parents = self.multi_vector_retriever.invoke(question)  # returns parent Documents
            # get their doc_ids
            parent_ids = [p.metadata.get("doc_id") for p in top_parents][:3]
            # filter child docs belonging to those parents
            candidate_children = [c for c in self.child_docs if c.metadata.get("doc_id") in parent_ids]
            # run dense similarity search within candidate_children by temporarily creating an in-memory vector store (cheap approach)
            # Simpler: score candidate_children by embedding similarity to query
            q_vec = self.embeddings.embed_query(question)
            # compute similarity (approx) by embedding candidate_children texts (expensive) -> instead use child_vectorstore but filter results by doc_id
            raw_hits = self.child_vectorstore.similarity_search(question, k=self.cfg.K*3)
            filtered = [d for d in raw_hits if d.metadata.get("doc_id") in parent_ids]
            docs = filtered[: self.cfg.K]
        elif strategy == "multi_vector":
            # retrieve via MultiVectorRetriever which maps summary/hypo vectors -> parent doc, and returns parent doc content
            parent_hits = self.multi_vector_retriever.invoke(question)
            # for each parent hit, optionally expand into child chunks (we append top child chunks)
            docs = []
            for p in parent_hits[: self.cfg.K]:
                pid = p.metadata.get("doc_id")
                # get top child chunks for the parent (use child_vectorstore with filter)
                child_hits = self.child_vectorstore.similarity_search(question, k=4)
                # filter
                child_for_parent = [c for c in child_hits if c.metadata.get("doc_id") == pid]
                docs.extend(child_for_parent)
            if not docs:
                # fallback to dense retriever global
                docs = self.dense_retriever.get_relevant_documents(question)
        else:
            # default naive
            docs = self.dense_retriever.get_relevant_documents(question)

        # answer generation
        answer_text, provenance = self._synthesize_answer(docs, question, prompt_template)

        # log experiment
        log_entry = {
            "question": question,
            "strategy": strategy,
            "prompt_version": prompt_version,
            "model_name": self.cfg.MODEL_NAME,
            "k_retrieved": len(docs),
            "retrieved_docs": [{"source": d.metadata.get("source_path") or d.metadata.get("file"), "metadata": d.metadata} for d in docs],
            "answer": answer_text
        }
        run_id = self.experiment_logger.log(log_entry)
        return {"run_id": run_id, "answer": answer_text, "provenance": provenance, "retrieved_count": len(docs)}

    # ---------- Evaluation helpers ----------
    @staticmethod
    def normalize_answer(s: str) -> str:
        s = s.lower().strip()
        import re
        s = re.sub(r"[^a-z0-9]+", " ", s)
        return " ".join(s.split())

    @staticmethod
    def exact_match(s1: str, s2: str) -> bool:
        return RAGSystem.normalize_answer(s1) == RAGSystem.normalize_answer(s2)

    @staticmethod
    def f1_score(pred: str, gold: str) -> float:
        p_tokens = RAGSystem.normalize_answer(pred).split()
        g_tokens = RAGSystem.normalize_answer(gold).split()
        if not p_tokens or not g_tokens:
            return 0.0
        common = set(p_tokens) & set(g_tokens)
        if not common:
            return 0.0
        prec = len(common) / len(p_tokens)
        rec = len(common) / len(g_tokens)
        if prec + rec == 0:
            return 0.0
        return 2 * prec * rec / (prec + rec)

    def evaluate_dataset(self, qa_pairs: List[Dict[str, str]], strategy: str = "auto", prompt_id: Optional[str] = None) -> Dict[str, Any]:
        results = []
        for q in qa_pairs:
            out = self.run_query(q["question"], strategy=strategy, prompt_id=prompt_id)
            pred = out["answer"]
            em = 1 if self.exact_match(pred, q["answer"]) else 0
            f1 = self.f1_score(pred, q["answer"])
            results.append({"question": q["question"], "gold": q["answer"], "pred": pred, "em": em, "f1": f1, "run_id": out["run_id"]})
        avg_em = sum(r["em"] for r in results) / len(results)
        avg_f1 = sum(r["f1"] for r in results) / len(results)
        return {"results": results, "avg_em": avg_em, "avg_f1": avg_f1}


# -------------------------
# Example usage (script)
# -------------------------
def main_example():
    cfg = RAGConfig()
    rag = RAGSystem(cfg)

    # ingest hierarchical json directory (expects json_data/dbe_<TICKER>/*.json)
    rag.ingest_documents(cfg.JSON_INPUT_ROOT)

    # build indexes / retrievers
    rag.build_indexes(persist=True)

    # register a prompt version
    default_prompt = (
        "You are an expert financial assistant. Use the context to answer the question.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer (cite sources as [Source X]):"
    )
    pid = rag.prompt_manager.register_prompt("fin_v1", default_prompt, "Default financial response prompt")

    # run an example query
    q = "What were the net interest expenses for MMM in 2024 and 2023?"
    out = rag.run_query(q, strategy="auto", prompt_id=pid)
    print("Run ID:", out["run_id"])
    print("Answer:\n", out["answer"])
    print("Provenance (first 3):", out["provenance"][:3])

if __name__ == "__main__":
    main_example()
